# Indexing SciDocs by OpenSearch for BM25 Model

- [beir/scidocs](https://ir-datasets.com/beir.html#beir/scidocs)

In [ ]:
import sys
!{sys.executable} -m pip install -q ir_datasets pandas opensearch-py dotenv

In [ ]:
import pprint
from tqdm import tqdm

Your opensearch password should be available in `~/.env`

```bash
    OPENSEARCH_INITIAL_ADMIN_PASSWORD="strong password"
```

In [ ]:
import os
from dotenv import load_dotenv
from opensearchpy import OpenSearch

load_dotenv()
host = 'localhost'
port = 9200
password = os.getenv("OPENSEARCH_INITIAL_ADMIN_PASSWORD")

client = OpenSearch(
    hosts=[{"host": host, "port": port}],
    http_auth=("admin", password),
    http_compress=True,
    use_ssl=True,
    verify_certs=False,
    ssl_assert_hostname=False,
    ssl_show_warn=False
)
pprint.pprint(client.info())

### Index a Corpus for BM25 Model

In [ ]:
import ir_datasets
dataset_name = "beir/scidocs"
dataset = ir_datasets.load(dataset_name)

In [ ]:
index_name = "scidocs_bm25"

In [ ]:
# Delete an existing index (be careful)
if client.indices.exists(index=index_name):
    response = client.indices.delete(index=index_name)
    pprint.pprint(response)
else:
    print(f"{index_name} does not exist")

In [ ]:
index_body = {
  "settings": {
    "index": {
      "number_of_shards": 1,
      "number_of_replicas": 0
    }
    # English corpus: rely on OpenSearch's default (standard) analyzer.
  },
  "mappings": {
    "properties": {
        "docid": { "type": "keyword" },
        "title": { "type": "text" },
        "text": { "type": "text" },
    }
  }
}

response = client.indices.create(index=index_name, body=index_body)
pprint.pprint(response)

Indexing

In [ ]:
def prepare_documents(dataset):
    """
    Prepare individual documents for indexing, with progress tracking.
    scidocs exposes title/text as separate fields, so no splitting is needed.
    """
    total_docs = dataset.docs_count()
    progress = tqdm(total=total_docs, desc="Indexing Documents")

    for doc in dataset.docs_iter():
        yield {
            "_id": doc.doc_id,  # Unique identifier for the document
            "_source": {
                "docid": doc.doc_id,
                "title": doc.title,
                "text": doc.text
            }
        }
        progress.update(1)  # Update progress bar

    progress.close()  # Close the progress bar

In [ ]:
from opensearchpy.helpers import bulk
success, failed = bulk(client, prepare_documents(dataset), index=index_name)
print(f"indexed: {success},  failed: {len(failed)}")